# Retrocausality Mini Project

**当前规模**：3 agents, 5×5 grid, 7 steps, Mesa 2.3.0, PyTorch TCN-based

目标：  
Agent 0 使用 TCN 预测未来碰撞 → 实时调整当前移动，避免碰撞；  
其他两个 agent 随机移动（allow_collisions=True 时倾向碰撞）。

主要功能：
- 生成混合数据（易碰撞 + 难碰撞）
- 训练 / 加载 TCN
- 对比：纯随机 vs 有 TCN 规则 的碰撞次数

In [5]:
# ==============================================
# 设置工作目录和路径（运行在 notebook 中时必需）
# ==============================================
import os
import sys
from pathlib import Path

# 在 Jupyter notebook 中获取项目根目录
# notebook 在 notebooks/ 子目录中，需要上升一级
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"工作目录已设置为: {os.getcwd()}")

工作目录已设置为: e:\projects\Dynamic-Retrocausal-Simulator


In [ ]:
# ==============================================
# 导入（只保留当前 mini 项目真正需要的）
# ==============================================
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
from tqdm import tqdm

from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation

# 项目自己的模块（假设你已按之前建议重构好）
from config import *
from src.model import RetroAgent, RetroModel, TCN
from src.tcn import train_tcn, load_tcn, predict_collision
from src.data_gen import collect_mixed_data
from src.evaluate import compare_rule_effects

# 固定随机种子，便于复现
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True

ModuleNotFoundError: No module named 'simulation_core'

In [2]:
import mesa
import torch
import pandas as pd
import tqdm
print(mesa.__version__)          # 应输出 2.3.0
print(torch.__version__)
print(pd.__version__)
print(tqdm.__version__)

2.3.0
2.7.0
3.0.1
4.67.3


In [3]:
print("STEPS:", STEPS)                # 7
print("SEQ_LEN:", SEQ_LEN)            # 5
print("device:", device)              # cuda or cpu
print("模型保存路径:", TCN_MODEL_PATH)

NameError: name 'STEPS' is not defined

## 1. 生成小规模测试数据（调试用）

In [ ]:
# 先跑一个小批量，检查数据生成是否正常
small_data, total_coll, save_path = collect_mixed_data(
    num_sims_collision_prone=30,
    num_sims_collision_free=30,
    steps=STEPS
)

print(f"小批量生成完成：总碰撞数 {total_coll}，保存到 {save_path}")
print(f"数据条目数：{len(small_data)}")

## 2. 检查单次模拟轨迹（调试用）

In [ ]:
# 手动跑一次模拟，看 Agent 0 是否使用了 TCN 规则
model = RetroModel(allow_collisions=True, tcn_path="models/tcn_best.pth")
data, coll_count = model.run_simulation(steps=7)

print(f"本次模拟 Agent 0 碰撞次数：{coll_count}")

# 打印 Agent 0 的移动序列
agent0_moves = [d['pos'] for d in data if d['agent_id'] == 0]
print("Agent 0 位置序列：", agent0_moves)

## 3. 对比实验（核心验证 retrocausality）

In [ ]:
# 小规模对比（快速看效果）
no_rule_coll, with_rule_coll = compare_rule_effects(
    num_sims=50,
    steps=STEPS,
    tcn_path="models/tcn_best.pth"   # 改成你最新的模型路径
)

reduction = (no_rule_coll - with_rule_coll) / no_rule_coll * 100 if no_rule_coll > 0 else 0
print(f"\n碰撞减少比例：{reduction:.1f}%")

# 如果效果明显，可以再跑大批量
# compare_rule_effects(num_sims=200, steps=STEPS, tcn_path=...) 

## 4. 快速检查 TCN 预测（调试模型）

In [ ]:
# 加载模型并测试单条输入
tcn = load_tcn("models/tcn_best.pth")

# 假设你有某条序列数据（形状 [1, SEQ_LEN, features]）
# 这里用随机假数据演示，实际请替换成你的真实输入
dummy_input = torch.randn(1, SEQ_LEN, 18).to(device)  # 改成你的实际特征维度

with torch.no_grad():
    pred = tcn(dummy_input)
    print("TCN 示例输出：", pred)
    
    # 如果是二分类碰撞预测
    prob = torch.sigmoid(pred).item()
    print(f"预测碰撞概率：{prob:.3f}")